In [6]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
print(f"GPU available: {torch.cuda.is_available()}")


GPU available: True


In [7]:
# read csv
df = pd.read_csv("dataset/vibe_coding_search.csv")
# drop empty text and duplicate text to preserve comments that dh ID
df = df.dropna(subset=['Text'])
df = df.drop_duplicates(subset=['Text'])

df['Clean_Text'] = df['Text'].astype(str).str.lower()

print(f"Total number of unique records {len(df)}")

Total number of unique records 75187


In [8]:
if 'Clean_Text' not in df.columns:
    df['Clean_Text'] = df['Text'].astype(str).str.lower()
    
job_keywords = [
    'hiring', 'hire', 'apply now', 'salary', 'years of experience', 
    'job requirement', 'qualifications', 'full-time', 'benefits', 
    'recruiting', 'competitive pay', 'join our team', 'AI Accelerator',
    'HOWTO', 'howto', 'iptv', 'IPTV'
]

pattern = '|'.join(job_keywords)

df = df[~df['Clean_Text'].str.contains(pattern, case=False, na=False)]

print(f"Records remaining after Keyword Blacklist: {len(df)}")

Records remaining after Keyword Blacklist: 71819


In [9]:
print(df.columns.to_list())
df.head(1)

['Source', 'ID', 'Type', 'Author', 'Text', 'Score', 'Date', 'Clean_Text']


,Source,ID,Type,Author,Text,Score,Date,Clean_Text
0,r/u_allstacksai,1qxiaje,Post,allstacksai,1B GitHub commits in 2025 - why the AI coding ...,1,2026-02-06,1b github commits in 2025 - why the ai coding ...


In [ ]:
# apply new schema and parentID logic
df['Source'], df['Title'] = 'Reddit', None
df['Word_Count'] = df['Clean_Text'].astype(str).apply(lambda x: len(x.split()))

# Sequential ID inference
post_ids = df['ID'].where(df['Type'].str.lower() == 'post', np.nan)
forward_filled_parents = post_ids.ffill()

# create parentID column
df['Parent_ID'] = np.where(df['Type'].str.lower() == 'post', None, forward_filled_parents)

missing_ids = df['ID'].isna()
if missing_ids.any():
    df.loc[missing_ids, 'ID'] = [f"comment_{i}" for i in range(missing_ids.sum())]

# enforce new schema 
target_schema = ['ID', 'Parent_ID', 'Source', 'Type', 'Author', 'Text', 'Score', 'Date', 'Clean_Text', 'Word_Count', 'Title']
df = df[target_schema]

print(f"Total unique records after cleaning: {len(df)}")
print(df.columns.tolist())

Total unique records after cleaning: 71819
['ID', 'Parent_ID', 'Source', 'Type', 'Author', 'Text', 'Score', 'Date', 'Clean_Text', 'Word_Count', 'Title']


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Ideal sentence 
target_query = (
    "vibe coding and natural language programming, giving in to the vibes with Cursor Composer or Windsurf Cascade, "
    "accepting AI code without reading diffs, rapid prototyping for non-technical founders, "
    "the death of junior developer roles, imposter syndrome and AI skill rot, "
    "spaghetti code and technical debt from LLMs, revolutionary productivity vs security risks and vulnerabilities, "
    "autonomous coding agents like Cline and Roo Code, coding beyond human comprehension"
)
# Encode the target query into a mathematical vector space
target_embedding = model.encode(target_query, convert_to_tensor=True)

print("Model and target embedding loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1049.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and target embedding loaded!


In [ ]:
# Batched processing to be faster 
corpus_embeddings = model.encode(df['Clean_Text'].tolist(), batch_size=64, convert_to_tensor=True, show_progress_bar=True)

# cos similarity betw ideal sentence and text
cosine_scores = util.cos_sim(target_embedding, corpus_embeddings)[0]
df['Relevance_Score'] = cosine_scores.cpu().numpy()

print("\nTop 3 Most Relevant Posts:")
print(df.nlargest(3, 'Relevance_Score')[['Text', 'Relevance_Score']])

Batches:  28%|██▊       | 315/1123 [00:12<00:28, 28.72it/s]

In [ ]:
def get_total_words(dataframe):
    return dataframe['Clean_Text'].apply(lambda x: len(str(x).split())).sum()

threshold = 0.40
step = 0.02
success = False

while threshold >= 0.10:
    # Filter the dataframe based on the current threshold
    df_filtered = df[df['Relevance_Score'] >= threshold]
    
    total_records = len(df_filtered)
    total_words = get_total_words(df_filtered)
    
    # Check assignment constraints
    if total_records >= 40000 and total_words >= 100000:
        print(f"Success! Extracted threshold: {threshold:.2f}")
        print(f"Final Records: {total_records} (Target: 40,000+)")
        print(f"Final Words: {total_words} (Target: 100,000+)")
        success = True
        break
    else:
        # if fails, threshold must be lowered
        print(f"Threshold {threshold:.2f} only yielded {total_records} records. Lowering...")
        threshold -= step

if not success:
    print(f"\nWARNING: Target of 40k records not met.")
    print(f"Saving the best available at 0.10 threshold: {len(df_filtered)} records.")

Threshold 0.40 only yielded 4402 records. Lowering...
Threshold 0.38 only yielded 5690 records. Lowering...
Threshold 0.36 only yielded 7159 records. Lowering...
Threshold 0.34 only yielded 8894 records. Lowering...
Threshold 0.32 only yielded 10728 records. Lowering...
Threshold 0.30 only yielded 12740 records. Lowering...
Threshold 0.28 only yielded 15007 records. Lowering...
Threshold 0.26 only yielded 17403 records. Lowering...
Threshold 0.24 only yielded 20009 records. Lowering...
Threshold 0.22 only yielded 22897 records. Lowering...
Threshold 0.20 only yielded 25916 records. Lowering...
Threshold 0.18 only yielded 29267 records. Lowering...
Threshold 0.16 only yielded 32970 records. Lowering...
Threshold 0.14 only yielded 37005 records. Lowering...
✅ Success! Extracted threshold: 0.12
Final Records: 41374 (Target: 40,000+)
Final Words: 7357620 (Target: 100,000+)


In [ ]:
df_final = df_filtered.drop(columns=['Relevance_Score'])

json_path = "dataset/vibe_coding_transformer_processed.json"
df_final.to_json(json_path, orient="records", indent=4)

print(f"Successfully saved {len(df_final)} records to {json_path}")

✅ Successfully saved 41374 records to dataset/vibe_coding_transformer_processed.json
